In [1]:
##### Isolated agriculutre's portion of FAO capital stock data 
# method described in data porcessing memo

import os
import pandas as pd
import numpy as np

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
value_added = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/FAO_value_added.csv")
capital_stock = pd.read_csv(f"{cd}/Data/Clean/Capital_stock/FAO_capital_stock.csv")

country_codes = pd.read_csv(
    f"{cd}/Data/Correspondence_tables/country_names.csv",
    encoding="cp1252"
)

# Set save path
save_path = f"{cd}/Data/Clean/Capital_stock/FAO_capital_stock_adjusted.csv"


In [3]:
### Fill in missing value added data 
# for 2020 using closest avialable year

# convert to long
col_to_keep = ['ISO3', '1970', '1971', '1972', '1973', '1974',
       '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983',
       '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992',
       '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001',
       '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010',
       '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019',
       '2020', '2021', '2022', '2023']
value_added = value_added[col_to_keep]

value_added_long = value_added.melt(id_vars='ISO3', var_name='Year', value_name='ag_value_added_share')
value_added_long['Year'] = value_added_long['Year'].astype(int)

# fill in using latest available year 
value_added_long = value_added_long.sort_values(['ISO3', 'Year'])

# Forward fill and backward fill
value_added_long['Value_ffill'] = value_added_long.groupby('ISO3')['ag_value_added_share'].ffill()
value_added_long['Value_bfill'] = value_added_long.groupby('ISO3')['ag_value_added_share'].bfill()

# Choose the closest
value_added_long['ag_value_added_share_filled'] = value_added_long.apply(
    lambda row: row['Value_ffill'] if pd.notna(row['Value_ffill']) and 
                               (pd.isna(row['Value_bfill']) or abs(row['Year'] - row['Year']) <= abs(row['Year'] - row['Year'])) 
    else row['Value_bfill'], axis=1
)

# Isolate 2020
value_added_long = value_added_long[value_added_long['Year'] == 2020]

col_to_keep = ['ISO3', 'ag_value_added_share_filled']
value_added_long = value_added_long[col_to_keep]

In [4]:
### Get country-level assumptions 
# for country's missing 2020 data
# using sub-regional (i.e. North Africa) average where available and regional (i.e. Africa) where not

# Filter country list to those in FAO data
country_codes = country_codes.merge(capital_stock['ISO3'], on='ISO3', how='right')

# Merge with ag share data
ag_shares = value_added_long.merge(country_codes, on='ISO3', how='right')

# get sub-region data
sub_region_count = (
    ag_shares
    .groupby('UN_subregion')['ag_value_added_share_filled']
    .count()
    .reset_index()
    .rename(columns={'ag_value_added_share_filled': 'sub_region_non_missing_count'})
)

sub_region_mean = (
    ag_shares
    .groupby('UN_subregion')['ag_value_added_share_filled']
    .mean()
    .reset_index()
    .rename(columns={'ag_value_added_share_filled': 'sub_region_mean'})
)

ag_shares = ag_shares.merge(sub_region_count, on='UN_subregion', how='left')
ag_shares = ag_shares.merge(sub_region_mean, on='UN_subregion', how='left')

# Count number of regions with data
region_count = (
    ag_shares
    .groupby('UN_region')['ag_value_added_share_filled']
    .count()
    .reset_index()
    .rename(columns={'ag_value_added_share_filled': 'region_non_missing_count'})
)

region_mean = (
    ag_shares
    .groupby('UN_region')['ag_value_added_share_filled']
    .mean()
    .reset_index()
    .rename(columns={'ag_value_added_share_filled': 'region_mean'})
)

ag_shares = ag_shares.merge(region_count, on='UN_region', how='left')
ag_shares = ag_shares.merge(region_mean, on='UN_region', how='left')

# fill with sub-regional average if sub-regional count is > 1 and with regional average if sub_regional count = 1
# except for Oceania - fill missing with sub-regional average for South East Asia

def fill_value(row): 
    # If it already has a value, keep it
    if pd.notna(row['ag_value_added_share_filled']):
        return row['ag_value_added_share_filled']
    
    # If sub-regional count >1 
    if row['sub_region_non_missing_count'] > 1:
        return row['sub_region_mean']
    
    # If sub-regional count <=1
    else: 
        return row['region_mean']

ag_shares['ag_value_added_share'] = ag_shares.apply(fill_value, axis=1)

# replace Melanesia, Micronesia, and Polynesia with South-eastern Asia

SEA_mean = sub_region_mean.loc[
    sub_region_mean['UN_subregion'] == 'South-eastern Asia', 
    'sub_region_mean'
].values[0]

override_subregions = ['Melanesia', 'Micronesia', 'Polynesia']

ag_shares['ag_share_final'] = np.where(
    ag_shares['UN_subregion'].isin(override_subregions), 
    SEA_mean,  
    ag_shares['ag_value_added_share']  
)

# keep only needed vars 
col_to_keep = ['ISO3', 'ag_share_final']
ag_shares = ag_shares[col_to_keep]

In [5]:
### Break out capital for ag only

# merge with capital stock data
capital_stock = capital_stock.merge(ag_shares, on='ISO3', how='outer')

# keep only 2020 data
col_to_keep = ['ISO3', 'Country_Name', '2020', 'ag_share_final']
capital_stock = capital_stock[col_to_keep]

capital_stock = capital_stock.rename(columns={
    '2020': 'AFF_capital_stock_mil_USD_nominal', 
    'ag_share_final': 'ag_share_AFF_value_added_2020'}
    )

# calculate ag capital stock
capital_stock['ag_capital_stock_mil_USD_nominal'] = capital_stock['AFF_capital_stock_mil_USD_nominal'] * capital_stock['ag_share_AFF_value_added_2020']

col_to_keep = ['ISO3', 'ag_capital_stock_mil_USD_nominal']
capital_stock = capital_stock[col_to_keep]

In [6]:
# Save
capital_stock.to_csv(save_path, index=False)